In [47]:
# Label Definitions
labels_map = {
    0: "T-Shirt/top",
    1: "Trouser",
    2: "Pullover",
    3: "Dress",
    4: "Coat",
    5: "Sandal",
    6: "Shirt",
    7: "Sneaker",
    8: "Bag",
    9: "Ankle Boot",
}

In [48]:
# Downloading and pre-processing the Fashion MNIST dataset
import torch
from torch.utils.data import Dataset
from torchvision import datasets
from torchvision.transforms import v2
import matplotlib.pyplot as plt

# In the transform, image is converted to PyTorch tensor. Convert pixel values from 0-255 to 0-1.
training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
)
test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
)

# Loading the data in batches
from torch.utils.data import DataLoader
train_dataloader = DataLoader(training_data, batch_size=64, shuffle=True)
test_dataloader = DataLoader(test_data, batch_size=64, shuffle=False)

# Generates iterator over batches
# train_features, train_labels = next(iter(train_dataloader))

In [49]:
# Selecting GPU
import os
from torch import nn
from torchvision import transforms

device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using cuda device


In [50]:
# Defining neural network
# class NeuralNetwork(nn.Module):
#     def __init__(self):
#         super().__init__()
#         self.flatten = nn.Flatten()
#         self.linear_relu_stack = nn.Sequential(
#             nn.Linear(28*28, 1024),
#             nn.ReLU(),
#             nn.Linear(1024, 512),
#             nn.ReLU(),
#             nn.Linear(512, 10),
#         )

#     def forward(self, x):
#         x = self.flatten(x)
#         logits = self.linear_relu_stack(x)
#         return logits

class CNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            # Block 1
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),

            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.ReLU(),

            nn.MaxPool2d(kernel_size=2),

            # Block 2
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),

            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(),

            nn.MaxPool2d(kernel_size=2),

            # Classification Head
            nn.Flatten(),

            nn.Linear(64 * 7 * 7, 256),
            nn.ReLU(),

            nn.Linear(256, 128),
            nn.ReLU(),

            nn.Linear(128, 10)
        )

    def forward(self, x):
        return self.network(x)

# Move model to device
# model = NeuralNetwork().to(device)
model= CNN().to(device)
print(model)

CNN(
  (network): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU()
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU()
    (7): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU()
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Flatten(start_dim=1, end_dim=-1)
    (11): Linear(in_features=3136, out_features=256, bias=True)
    (12): ReLU()
    (13): Linear(in_features=256, out_features=128, bias=True)
    (14): ReLU()
    (15): Linear(in_features=128, out_features=10, bias=True)
  )
)


In [35]:
# Model Verification Demo Test
# X = torch.rand(1, 28, 28, device=device)
# logits = model(X)
# pred_probab = nn.Softmax(dim=1)(logits)
# y_pred = pred_probab.argmax(1)
# print(f"Predicted class: {y_pred}")

In [51]:
def train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    # Set the model to training mode - important for batch normalization and dropout layers
    # Unnecessary in this situation but added for best practices
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        # Compute prediction and loss
        X, y = X.to(device), y.to(device)
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), batch * batch_size + len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")


def test_loop(dataloader, model, loss_fn):
    # Set the model to evaluation mode - important for batch normalization and dropout layers
    # Unnecessary in this situation but added for best practices
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0

    # Evaluating the model with torch.no_grad() ensures that no gradients are computed during test mode
    # also serves to reduce unnecessary gradient computations and memory usage for tensors with requires_grad=True
    with torch.no_grad():
        for X, y in dataloader:
          X, y = X.to(device), y.to(device)
          pred = model(X)
          test_loss += loss_fn(pred, y).item()
          correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

In [52]:
learning_rate = 1e-3
batch_size = 64

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

epochs = 10
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train_loop(train_dataloader, model, loss_fn, optimizer)
    test_loop(test_dataloader, model, loss_fn)
print("Done!")

Epoch 1
-------------------------------
loss: 2.317709  [   64/60000]
loss: 0.624799  [ 6464/60000]
loss: 0.617439  [12864/60000]
loss: 0.719626  [19264/60000]
loss: 0.443772  [25664/60000]
loss: 0.394604  [32064/60000]
loss: 0.523092  [38464/60000]
loss: 0.338384  [44864/60000]
loss: 0.250012  [51264/60000]
loss: 0.331188  [57664/60000]
Test Error: 
 Accuracy: 88.1%, Avg loss: 0.321218 

Epoch 2
-------------------------------
loss: 0.256759  [   64/60000]
loss: 0.196281  [ 6464/60000]
loss: 0.358892  [12864/60000]
loss: 0.239763  [19264/60000]
loss: 0.252361  [25664/60000]
loss: 0.191333  [32064/60000]
loss: 0.324234  [38464/60000]
loss: 0.225616  [44864/60000]
loss: 0.273033  [51264/60000]
loss: 0.352814  [57664/60000]
Test Error: 
 Accuracy: 89.4%, Avg loss: 0.285994 

Epoch 3
-------------------------------
loss: 0.147994  [   64/60000]
loss: 0.119286  [ 6464/60000]
loss: 0.146353  [12864/60000]
loss: 0.165052  [19264/60000]
loss: 0.142984  [25664/60000]
loss: 0.241954  [32064/600

In [53]:
import torchvision.models as models

torch.save(model.state_dict(), "fashion_mnist.pth")
# model = NeuralNetwork()
model = CNN()

model.load_state_dict(torch.load("fashion_mnist.pth"))
model.eval()

CNN(
  (network): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU()
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU()
    (7): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU()
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Flatten(start_dim=1, end_dim=-1)
    (11): Linear(in_features=3136, out_features=256, bias=True)
    (12): ReLU()
    (13): Linear(in_features=256, out_features=128, bias=True)
    (14): ReLU()
    (15): Linear(in_features=128, out_features=10, bias=True)
  )
)

In [54]:
from google.colab import files

uploaded = files.upload()

import zipfile
import io
import pandas as pd
from PIL import Image

import torch

# Load model
# model = NeuralNetwork().to(device)
model = CNN().to(device)
model.load_state_dict(torch.load("fashion_mnist.pth", map_location=device))
model.eval()

transform = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True)
])

predictions = []

with zipfile.ZipFile("test_set.zip", "r") as z:

    image_files = sorted(
        [f for f in z.namelist()
         if f.startswith("images/") and f.endswith(".png")]
    )

    image_files = sorted(
    image_files,
    key=lambda x: int(x.split("/")[-1].replace(".png", ""))
    )

    with torch.no_grad():
        for file in image_files:

            image = Image.open(io.BytesIO(z.read(file)))

            x = transform(image)
            x = x.unsqueeze(0).to(device)

            logits = model(x)
            pred = logits.argmax(dim=1).item()

            image_id = int(file.split("/")[-1].replace(".png", ""))

            predictions.append([image_id, pred])

submission = pd.DataFrame(
    predictions,
    columns=["image_id", "label"]
)

submission = submission.sort_values("image_id")

submission.to_csv("submission.csv", index=False)

print(submission.head())
print("CSV saved as submission.csv")
files.download("submission.csv")

Saving test_set.zip to test_set (4).zip
   image_id  label
0         0      5
1         1      1
2         2      7
3         3      4
4         4      3
CSV saved as submission.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>